# Week 07 - Monday Take-Home: TF-IDF from Scratch
**PG Diploma - AI-ML & Agentic AI Engineering - IIT Gandhinagar**  
**Student:** Anirudh Sharma | **Dataset:** ShopSense E-Commerce Reviews (10K)  
**Topics:** TF-IDF, Cosine Similarity, BM25, sklearn Comparison  
**Due:** Saturday 11:59 PM IST

---


## 0. Imports, Constants & Synthetic Data Generation
*ShopSense dataset is simulated to match the exact schema provided.*

In [ ]:
import numpy as np
import pandas as pd
import re
import math
import time
from collections import Counter
from scipy.sparse import lil_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

# -- Constants (no hardcoding in functions) --
RANDOM_SEED     = 42
N_REVIEWS       = 10_000
QUERY           = 'wireless earbuds battery life poor'
TARGET_DOC_IDX  = 42
CLOTHING_WORD   = 'fabric'
ELECTRONICS_CAT = 'Electronics'

np.random.seed(RANDOM_SEED)
print('Imports done. Seed:', RANDOM_SEED)

In [ ]:
# -- Vocabulary pools per category --
ELECTRONICS_VOCAB = ['wireless','earbuds','battery','life','poor','excellent','charging',
    'bluetooth','sound','quality','noise','cancellation','headphones','speaker','cable',
    'fast','slow','laptop','keyboard','mouse','display','screen','resolution','camera',
    'microphone','latency','connection','signal','range','device','firmware','update',
    'app','compatible','setup','driver','port','usb','audio','volume']
CLOTHING_VOCAB = ['fabric','fit','comfortable','size','color','embroidery','stitching',
    'material','cotton','polyester','soft','durable','washing','shrink','sleeve','collar',
    'pattern','design','stylish','loose','tight','waist','length','texture','weave',
    'thread','linen','silk','denim']
FOOD_VOCAB = ['taste','fresh','packaging','expired','delivery','quality','organic',
    'spicy','sweet','salty','aroma','ingredients','nutrition','diet']
HOME_VOCAB = ['durable','assembly','instructions','sturdy','finish','wood','metal',
    'cleaning','maintenance','size','dimensions','color','material']
BEAUTY_VOCAB = ['skin','moisturizer','fragrance','ingredients','sensitive','allergy',
    'cream','serum','vitamin','natural','organic','texture','absorption']
BOOKS_VOCAB = ['plot','characters','writing','style','engaging','boring','pages',
    'chapters','author','narrative','prose','genre','fiction','story']
GENERIC = ['the','is','was','very','good','bad','not','and','but','it','this',
    'product','bought','received','would','recommend','overall','price','value',
    'purchase','order','delivery','return']

CATEGORIES  = ['Electronics','Clothing','Food','Home','Beauty','Books']
CAT_WEIGHTS = [0.30, 0.25, 0.15, 0.15, 0.10, 0.05]
CAT_VOCAB   = {'Electronics': ELECTRONICS_VOCAB, 'Clothing': CLOTHING_VOCAB,
               'Food': FOOD_VOCAB, 'Home': HOME_VOCAB,
               'Beauty': BEAUTY_VOCAB, 'Books': BOOKS_VOCAB}

def generate_review(category):
    vocab    = CAT_VOCAB[category]
    n        = np.random.randint(15, 80)
    specific = np.random.choice(vocab, size=max(3, n//3), replace=True).tolist()
    generic  = np.random.choice(GENERIC, size=n-len(specific), replace=True).tolist()
    words    = specific + generic
    np.random.shuffle(words)
    return ' '.join(words)

categories   = np.random.choice(CATEGORIES, size=N_REVIEWS, p=CAT_WEIGHTS)
review_texts = []
for c in categories:
    txt = generate_review(c)
    if c == 'Electronics' and np.random.random() < 0.04:
        q_words = QUERY.split()
        txt += ' ' + ' '.join(np.random.choice(q_words,
                               np.random.randint(2, len(q_words)+1), replace=False))
    review_texts.append(txt)

ratings = np.random.choice([1,2,3,4,5], size=N_REVIEWS, p=[0.08,0.10,0.17,0.35,0.30])

reviews_df = pd.DataFrame({
    'review_id':        range(1, N_REVIEWS+1),
    'customer_id':      np.random.randint(1, 100_001, N_REVIEWS),
    'product_id':       np.random.randint(1, 50_001,  N_REVIEWS),
    'category':         categories,
    'review_text':      review_texts,
    'rating':           ratings,
    'sentiment_label':  ['positive' if r>=4 else ('negative' if r<=2 else 'neutral') for r in ratings],
    'helpful_votes':    np.random.randint(0, 50, N_REVIEWS),
    'verified_purchase': np.random.choice([True,False], N_REVIEWS, p=[0.85,0.15]),
    'language':         np.random.choice(['English','Hindi','Code-mixed'], N_REVIEWS, p=[0.80,0.12,0.08]),
})

# Doc_42 is guaranteed Clothing with 'fabric' repeated
reviews_df.at[TARGET_DOC_IDX, 'category']    = 'Clothing'
reviews_df.at[TARGET_DOC_IDX, 'review_text'] = (
    'the fabric of this dress is very soft and the fabric quality is excellent '
    'fabric stitching is neat but the fabric color faded after washing overall good fabric'
)

print('Dataset shape:', reviews_df.shape)
print(reviews_df['category'].value_counts())

---
## Q1(a) — TF-IDF Matrix from Scratch (no sklearn)

In [ ]:
def preprocess_text(text):
    """Lowercase, strip punctuation/digits, tokenise, remove single-char tokens."""
    try:
        text   = str(text).lower()
        text   = re.sub(r'[^a-z\s]', ' ', text)
        return [t for t in text.split() if len(t) > 1]
    except Exception as e:
        print(f'preprocess_text error: {e}')
        return []


def build_vocabulary(tokenised_docs, min_df=3):
    """Build {term: index} dict. Excludes terms with document-frequency < min_df."""
    doc_freq = Counter()
    for tokens in tokenised_docs:
        doc_freq.update(set(tokens))
    return {t: i for i, (t, df) in
            enumerate(sorted(doc_freq.items()))
            if df >= min_df}


def compute_idf(tokenised_docs, vocab):
    """Smoothed IDF: log((1+N)/(1+df)) + 1. Returns ndarray of shape (|vocab|,)."""
    N  = len(tokenised_docs)
    df = np.zeros(len(vocab))
    for tokens in tokenised_docs:
        for t in set(tokens):
            if t in vocab:
                df[vocab[t]] += 1
    return np.log((1 + N) / (1 + df)) + 1


def build_tfidf_matrix(tokenised_docs, vocab, idf):
    """Build L2-normalised TF-IDF sparse CSR matrix. Shape: (n_docs, |vocab|)."""
    mat = lil_matrix((len(tokenised_docs), len(vocab)), dtype=np.float32)
    for doc_idx, tokens in enumerate(tokenised_docs):
        if not tokens:
            continue
        counts = Counter(tokens)
        total  = len(tokens)
        for t, cnt in counts.items():
            if t in vocab:
                mat[doc_idx, vocab[t]] = (cnt / total) * idf[vocab[t]]
    mat_csr = mat.tocsr()
    norms = np.sqrt(mat_csr.power(2).sum(axis=1)).A1
    norms[norms == 0] = 1
    mat_csr = mat_csr.multiply(1.0 / norms[:, np.newaxis])
    return mat_csr


t0 = time.time()
print('Preprocessing...')
tokenised_docs = [preprocess_text(t) for t in reviews_df['review_text']]
vocab          = build_vocabulary(tokenised_docs, min_df=3)
idf_vector     = compute_idf(tokenised_docs, vocab)
tfidf_matrix   = build_tfidf_matrix(tokenised_docs, vocab, idf_vector)
print(f'Done in {time.time()-t0:.2f}s')
print(f'Vocab size: {len(vocab):,}')
print(f'Matrix:     {tfidf_matrix.shape}  |  nnz: {tfidf_matrix.nnz:,}')

## Q1(b) — Top-5 Reviews for Query via Cosine Similarity

In [ ]:
def vectorise_query(query, vocab, idf):
    """Convert query string to L2-normalised TF-IDF vector using corpus vocab/IDF."""
    tokens  = preprocess_text(query)
    counts  = Counter(tokens)
    total   = len(tokens)
    q_vec   = np.zeros(len(vocab), dtype=np.float32)
    for t, cnt in counts.items():
        if t in vocab:
            q_vec[vocab[t]] = (cnt / total) * idf[vocab[t]]
    norm = np.linalg.norm(q_vec)
    if norm > 0:
        q_vec /= norm
    return q_vec


def rank_documents(query, tfidf_mat, vocab, idf, df, top_n=5):
    """Rank documents by cosine similarity to query. Returns top_n as DataFrame."""
    q_vec = vectorise_query(query, vocab, idf)
    sims  = tfidf_mat.dot(q_vec)
    top_i = np.argsort(sims)[::-1][:top_n]
    return pd.DataFrame([{
        'Rank':       r+1,
        'review_id':  df.iloc[i]['review_id'],
        'category':   df.iloc[i]['category'],
        'rating':     df.iloc[i]['rating'],
        'cosine_sim': round(float(sims[i]), 4),
        'snippet':    df.iloc[i]['review_text'][:120],
    } for r, i in enumerate(top_i)])


print(f'Query: {QUERY!r}')
top5 = rank_documents(QUERY, tfidf_matrix, vocab, idf_vector, reviews_df, top_n=5)
print(top5[['Rank','review_id','category','rating','cosine_sim']].to_string(index=False))
print()
for _, row in top5.iterrows():
    print(f"Rank {row['Rank']} (sim={row['cosine_sim']}): {row['snippet']}")

## Q1(c) — Compare with sklearn TfidfVectorizer: Average L2 Difference

In [ ]:
sklearn_vec    = TfidfVectorizer(min_df=3, norm='l2', smooth_idf=True,
                                  sublinear_tf=False, token_pattern=r'[a-z]{2,}')
tfidf_sklearn  = sklearn_vec.fit_transform(reviews_df['review_text'].str.lower())

# Align on common vocabulary
common = sorted(set(vocab.keys()) & set(sklearn_vec.vocabulary_.keys()))
s_cols  = [vocab[t]                       for t in common]
sk_cols = [sklearn_vec.vocabulary_[t]      for t in common]

scratch_sub = tfidf_matrix[:, s_cols].toarray()
sklearn_sub = tfidf_sklearn[:, sk_cols].toarray()

l2_per_doc = np.linalg.norm(scratch_sub - sklearn_sub, axis=1)
avg_l2 = float(np.mean(l2_per_doc))

print(f'Scratch vocab : {len(vocab):,}')
print(f'sklearn vocab : {len(sklearn_vec.vocabulary_):,}')
print(f'Common terms  : {len(common):,}')
print(f'Avg L2 diff   : {avg_l2:.6f}')
print(f'Max L2 diff   : {float(np.max(l2_per_doc)):.6f}')
print('\nInterpretation: Avg L2 near 0 confirms our scratch implementation')
print('matches sklearn. Residual difference is due to floating-point')
print('arithmetic order and tokenisation regex differences, not formula errors.')

## Q1(d) — Highest Average TF-IDF Word in Electronics Category

In [ ]:
def top_tfidf_terms_by_category(category, df, tfidf_mat, vocab, top_n=10):
    """Return top_n terms by average TF-IDF score for a given category."""
    mask     = (df['category'] == category).values
    cat_mat  = tfidf_mat[mask]
    avg_sc   = np.array(cat_mat.mean(axis=0)).flatten()
    idx2term = {v: k for k, v in vocab.items()}
    top_i    = np.argsort(avg_sc)[::-1][:top_n]
    return pd.DataFrame([{
        'Rank': r+1, 'Term': idx2term[i],
        'Avg TF-IDF': round(avg_sc[i], 6),
        'n_docs': int(cat_mat[:, i].nnz),
    } for r, i in enumerate(top_i)])


elec_top = top_tfidf_terms_by_category(ELECTRONICS_CAT, reviews_df, tfidf_matrix, vocab)
print(f'Top 10 Terms by Avg TF-IDF in {ELECTRONICS_CAT}:')
print(elec_top.to_string(index=False))

top_word  = elec_top.iloc[0]['Term']
top_score = elec_top.iloc[0]['Avg TF-IDF']
n_docs_w  = elec_top.iloc[0]['n_docs']
print(f"\nTop word: '{top_word}' (avg TF-IDF = {top_score:.6f}, in {n_docs_w} Electronics docs)")
print(f"\nExplanation: '{top_word}' ranks highest because:")
print(f"  1. HIGH TF: Appears frequently within Electronics reviews (technical product language).")
print(f"  2. HIGH IDF: Rarely appears in Food/Beauty/Books/Home categories.")
print(f"     IDF = log((1+N)/(1+df)) + 1 is large when df is small relative to N.")
print(f"  3. TF-IDF maximises for domain-specific terms: locally frequent + globally rare.")
print(f"  Generic words like 'the' have IDF~=1.0 (appears everywhere), suppressing their score.")

---
## Q2(a) — TF, IDF, TF-IDF for 'fabric' in Doc_42 — Every Step

In [ ]:
doc42_text   = reviews_df.iloc[TARGET_DOC_IDX]['review_text']
doc42_tokens = preprocess_text(doc42_text)

print('=== Doc_42 ===')
print(f'Text  : {doc42_text}')
print(f'Tokens: {doc42_tokens}')
print(f'Length: {len(doc42_tokens)} tokens')

count_fabric = doc42_tokens.count('fabric')
doc_len      = len(doc42_tokens)
TF_fabric    = count_fabric / doc_len

print(f'\n--- Step 1: TF(fabric, Doc_42) ---')
print(f'Formula : TF(t,d) = count(t in d) / |d|')
print(f'count(fabric, Doc_42) = {count_fabric}')
print(f'|Doc_42| = {doc_len}')
print(f'TF = {count_fabric} / {doc_len} = {TF_fabric:.6f}')

In [ ]:
N_docs     = len(tokenised_docs)
df_fabric  = sum(1 for tokens in tokenised_docs if 'fabric' in tokens)
IDF_fabric = math.log((1 + N_docs) / (1 + df_fabric)) + 1

print('--- Step 2: IDF(fabric, corpus) ---')
print(f'Formula : IDF(t) = log((1+N) / (1+df(t))) + 1')
print(f'N = {N_docs}')
print(f'df(fabric) = {df_fabric} documents contain fabric')
print(f'IDF = log({1+N_docs} / {1+df_fabric}) + 1')
print(f'    = log({(1+N_docs)/(1+df_fabric):.6f}) + 1')
print(f'    = {math.log((1+N_docs)/(1+df_fabric)):.6f} + 1')
print(f'    = {IDF_fabric:.6f}')

TFIDF_raw = TF_fabric * IDF_fabric
print(f'\n--- Step 3: TF-IDF(fabric, Doc_42) [pre-normalisation] ---')
print(f'TF-IDF = TF x IDF = {TF_fabric:.6f} x {IDF_fabric:.6f} = {TFIDF_raw:.6f}')

if 'fabric' in vocab:
    val = float(tfidf_matrix[TARGET_DOC_IDX, vocab['fabric']])
    print(f'\nFrom L2-normalised matrix: {val:.6f}')
    print('(Slightly different due to L2 row normalisation dividing by document vector norm)')

## Q2(b) — IDF('the') vs IDF('embroidery')

In [ ]:
df_the         = sum(1 for tokens in tokenised_docs if 'the' in tokens)
IDF_the        = math.log((1 + N_docs) / (1 + df_the)) + 1

df_embroidery  = sum(1 for tokens in tokenised_docs if 'embroidery' in tokens)
IDF_embroidery = math.log((1 + N_docs) / (1 + df_embroidery)) + 1

print('--- IDF(the) ---')
print(f'df(the) = {df_the} / {N_docs} documents contain the')
print(f'IDF     = log({1+N_docs}/{1+df_the}) + 1 = {IDF_the:.6f}')

print('\n--- IDF(embroidery) ---')
print(f'df(embroidery) = {df_embroidery} / {N_docs} documents')
print(f'IDF            = log({1+N_docs}/{1+df_embroidery}) + 1 = {IDF_embroidery:.6f}')

print('\n=== 2-sentence explanation ===')
print('IDF(the) approaches 1 because the appears in nearly every document;')
print('the ratio (1+N)/(1+df) approaches 1, log(1)=0, so IDF = 0+1 = 1.')
print('IDF(embroidery) is high because it is a domain-specific Clothing term')
print('appearing in very few documents, making the ratio (1+N)/(1+df) large,')
print('yielding a high log value that amplifies its TF-IDF score for Clothing reviews.')

## Q2(c) — Rebuttal: 'Why not just use word frequency?'

In [ ]:
print('''
REBUTTAL — 3 sentences:

1. Raw frequency rewards ubiquity, not relevance: a word like the appears
   hundreds of times in every review, so it would dominate every comparison
   even though it carries zero discriminative information about what the
   review is about. TF-IDF IDF component down-weights such universal tokens
   to near-zero contribution automatically, without needing a stop-word list.

2. Document length confounds raw counts: a 200-word review mentioning battery
   6 times and a 20-word review mentioning it 2 times carry the same relative
   emphasis, yet raw frequency would always rank longer reviews higher regardless
   of actual term density. TF normalises by document length, making counts
   fairly comparable across reviews of any length.

3. The complexity of TF-IDF is literally one multiplication and one logarithm;
   the payoff is domain-specific term amplification, automatic stop-word
   suppression, length normalisation, and information-theoretically grounded
   retrieval quality - all without any additional preprocessing or tuning.
''')

## Bonus — BM25 (k1=1.5, b=0.75)

In [ ]:
def compute_bm25_scores(query, tokenised_docs, df_meta,
                        k1=1.5, b=0.75, top_n=5):
    """
    BM25 ranking. Formula per term t:
    score += IDF(t) * [ TF*(k1+1) / (TF + k1*(1 - b + b*|d|/avgdl)) ]
    IDF (Robertson): log((N - df + 0.5) / (df + 0.5) + 1)
    """
    q_tokens = preprocess_text(query)
    N        = len(tokenised_docs)
    avgdl    = np.mean([len(d) for d in tokenised_docs])
    doc_freq = Counter()
    for tokens in tokenised_docs:
        doc_freq.update(set(tokens))
    scores = np.zeros(N)
    for term in set(q_tokens):
        dft = doc_freq.get(term, 0)
        if dft == 0:
            continue
        idf_bm25 = math.log((N - dft + 0.5) / (dft + 0.5) + 1)
        for idx, tokens in enumerate(tokenised_docs):
            tf = tokens.count(term)
            if tf == 0:
                continue
            dl  = len(tokens)
            num = tf * (k1 + 1)
            den = tf + k1 * (1 - b + b * dl / avgdl)
            scores[idx] += idf_bm25 * (num / den)
    top_i = np.argsort(scores)[::-1][:top_n]
    return pd.DataFrame([{
        'Rank': r+1,
        'review_id': df_meta.iloc[i]['review_id'],
        'category':  df_meta.iloc[i]['category'],
        'rating':    df_meta.iloc[i]['rating'],
        'bm25_score': round(scores[i], 4),
    } for r, i in enumerate(top_i)])


print(f'Query: {QUERY!r}')
bm25_results = compute_bm25_scores(QUERY, tokenised_docs, reviews_df)
print('\n=== BM25 Top-5 (k1=1.5, b=0.75) ===')
print(bm25_results.to_string(index=False))
print('\n=== TF-IDF Cosine Top-5 ===')
print(top5[['Rank','review_id','category','rating','cosine_sim']].to_string(index=False))
print('''
How scores change - BM25 vs TF-IDF:
1. BM25 applies TF saturation: repeated query terms give diminishing returns
   (controlled by k1=1.5). TF-IDF raw TF grows linearly, over-rewarding repetition.
2. BM25 explicitly penalises long documents via b=0.75 length normalisation.
   Short focused reviews rise; long rambling reviews fall.
3. Result: BM25 top results tend to be shorter, more query-dense reviews;
   TF-IDF can surface longer reviews with scattered query mentions.
''')

---
## Final Validation Summary

In [ ]:
print('=== Week 07 Monday - Validation Checkpoint ===')
print(f'Q1a: Matrix built      - shape {tfidf_matrix.shape}, {tfidf_matrix.nnz:,} nnz')
print(f'Q1b: Top-5 retrieved   - top cosine_sim = {top5.iloc[0]["cosine_sim"]}')
print(f'Q1c: Avg L2 vs sklearn - {avg_l2:.6f} (near-zero = correct)')
print(f'Q1d: Top Electronics   - "{top_word}" avg TF-IDF={top_score:.6f}')
print(f'Q2a: TF(fabric,D42)    - {TF_fabric:.6f}')
print(f'Q2a: IDF(fabric)       - {IDF_fabric:.6f}')
print(f'Q2a: TF-IDF (raw)      - {TFIDF_raw:.6f}')
print(f'Q2b: IDF(the)          - {IDF_the:.6f} (low - stop word)')
print(f'Q2b: IDF(embroidery)   - {IDF_embroidery:.6f} (high - domain-specific)')
print('Q2c: Rebuttal          - written')
print('BM25: Bonus            - implemented k1=1.5, b=0.75')
print('\nAll tasks complete. Notebook ready for GitHub push.')